# Annual European electricity-mix validation

This notebook builds annual SHRECC consumption mixes for the European countries represented by both source datasets. Historical years use Energy Charts; prospective years use ENTSO-E TYNDP automatically.

The workflow deliberately processes one year at a time. A full European year contains 8,760 hourly systems, so retaining all five hourly result datasets simultaneously would use considerable memory. Each compact annual table and Brightway database is preserved before the yearly pipeline object is released.

In [ ]:
from functools import lru_cache
from gc import collect
from pathlib import Path

import bw2calc as bc
import bw2data as bd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from shrecc import NewDatabase

## Configuration

`COUNTRIES` is the end-to-end European model coverage shared by Energy Charts, TYNDP, and the REMIND-EU topology. It includes external European nodes represented in all three systems. Iceland is absent from the source models; Kosovo is present in TYNDP but currently lacks a REMIND-EU region assignment, so neither can yet be included in a harmonized five-year run.

Set `YEARS_TO_CREATE` to only the unfinished years when resuming an interrupted run. Annual tables for the other years are loaded from `OUTPUT_DIR`.

In [ ]:
YEARS = [2021, 2025, 2035, 2040, 2050]
YEARS_TO_CREATE = YEARS.copy()

COUNTRIES = [
    "AL", "AT", "BA", "BE", "BG", "CH", "CY", "CZ", "DE", "DK",
    "EE", "ES", "FI", "FR", "GR", "HR", "HU", "IE", "IT", "LT",
    "LU", "LV", "MD", "ME", "MK", "MT", "NL", "NO", "PL", "PT",
    "RO", "RS", "RU", "SE", "SI", "SK", "TR", "UA", "UK",
]

PROJECT_NAME = "SHRECCei311"
SCENARIO = "DE"
CLIMATE_YEAR = 2009

BACKGROUND_DATABASES = {
    2021: "ecoinvent-3.11-cutoff",
    2025: "ecoinvent-3.11-cutoff",
    2035: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2035 2026-07-10",
    2040: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2040 2026-07-10",
    2050: "ei_cutoff_3.11_remind-eu_SSP2-PkBudg650_2050 2026-07-10",
}
FOREGROUND_DATABASES = {
    year: f"shrecc_annual_europe_{year}" for year in YEARS
}
BACKGROUND_LABELS = {
    2021: "ecoinvent 3.11",
    2025: "ecoinvent 3.11",
    2035: "premise 2035",
    2040: "premise 2040",
    2050: "premise 2050",
}

CLIMATE_METHOD = (
    "ecoinvent-3.11",
    "EF v3.1",
    "climate change",
    "global warming potential (GWP100)",
)
METHOD_CONFIG = {"impact_categories": [CLIMATE_METHOD]}

OUTPUT_DIR = Path("export") / "annual_validation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

WRITE_DATABASES = True
COUNTRY_TO_PLOT = "IT"

In [ ]:
bd.projects.set_current(PROJECT_NAME)

missing_backgrounds = {
    year: name
    for year, name in BACKGROUND_DATABASES.items()
    if name not in bd.databases
}
if missing_backgrounds:
    raise ValueError(f"Missing background databases: {missing_backgrounds}")
if CLIMATE_METHOD not in bd.methods:
    raise ValueError(f"Brightway method is not installed: {CLIMATE_METHOD}")

display(
    pd.DataFrame(
        {
            "source": ["Energy Charts", "Energy Charts", "TYNDP", "TYNDP", "TYNDP"],
            "background": pd.Series(BACKGROUND_DATABASES),
            "foreground": pd.Series(FOREGROUND_DATABASES),
        },
        index=YEARS,
    ).rename_axis("year")
)

## Create annual mixes

No hour range is supplied: every hour from January 1 at 00:00 through December 31 at 23:00 is selected. These five years are not leap years, so each result must contain exactly 8,760 timestamps.

The cutoff is disabled for validation so the mapped shares can be compared without hiding small technologies in a residual category. `consumption_mix_volume` is omitted to avoid duplicating the largest result array; it is not required to write the normalized annual inventories.

`NewDatabase` currently aggregates a selected period as the arithmetic mean of its normalized hourly mixes. This notebook validates exactly that database-building behavior. A demand-weighted annual mix would answer a different question and should be introduced as an explicit aggregation option rather than silently substituted here.

In [ ]:
annual_tables = {}
run_records = []

for year in YEARS:
    table_path = OUTPUT_DIR / f"annual_mix_{year}.csv"

    if year not in YEARS_TO_CREATE:
        if not table_path.is_file():
            raise FileNotFoundError(
                f"No saved table for {year}; add it to YEARS_TO_CREATE"
            )
        annual_tables[year] = pd.read_csv(table_path, index_col=[0, 1, 2, 3])
        run_records.append(
            {
                "year": year,
                "source": "saved table",
                "hours": 8760,
                "activities": len(annual_tables[year]),
                "foreground database": FOREGROUND_DATABASES[year],
            }
        )
        continue

    print(f"\nCreating annual European mix for {year}")
    annual_database = NewDatabase(
        years=year,
        countries=COUNTRIES,
        project_name=PROJECT_NAME,
        bg_db_name=BACKGROUND_DATABASES[year],
        my_db_name=FOREGROUND_DATABASES[year],
        scenario=SCENARIO,
        climate_year=CLIMATE_YEAR,
        time_range=[
            f"{year}-01-01 00:00:00",
            f"{year}-12-31 23:00:00",
        ],
        cutoff=0,
        include_cutoff=False,
        strict=False,
        network=True,
        zero_consumption="month_hour_average",
        include_consumption_mix_volume=False,
        download=True,
        verbose=True,
    ).create()

    hour_count = annual_database.results(year).sizes["time"]
    if hour_count != 8760:
        raise ValueError(f"Expected 8760 hours for {year}; found {hour_count}")

    annual_tables[year] = annual_database.table(year).copy()
    annual_tables[year].to_csv(table_path)

    if WRITE_DATABASES:
        annual_database.write()

    run_records.append(
        {
            "year": year,
            "source": annual_database.sources[year],
            "hours": hour_count,
            "activities": len(annual_tables[year]),
            "foreground database": FOREGROUND_DATABASES[year],
        }
    )

    del annual_database
    collect()

run_summary = pd.DataFrame(run_records).set_index("year")
display(run_summary)

In [ ]:
annual_mix_checks = pd.concat(
    {
        year: table.sum(axis=0).rename("sum")
        for year, table in annual_tables.items()
    },
    names=["year", "country"],
).to_frame()
annual_mix_checks["difference from one"] = annual_mix_checks["sum"] - 1

if not np.allclose(annual_mix_checks["sum"], 1, atol=1e-8):
    display(annual_mix_checks.loc[~np.isclose(annual_mix_checks["sum"], 1, atol=1e-8)])
    raise ValueError("At least one annual mapped mix does not sum to one")

display(annual_mix_checks.groupby(level="year").agg(["min", "max"]))

## Compare technology mixes

The SHRECC tables contain mapped generating activities. For a composition-level comparison, the reference is therefore each background database's country-level `electricity, high voltage, production mix`, whose direct electricity inputs expose generating activities.

This comparison is intentionally separate from the LCIA reference below. LCIA uses `market for electricity, high voltage`, because the market is the conventional background supply product and includes the background model's own import and network structure.

In [ ]:
COUNTRY_ALIASES = {"UK": "GB", "NIE": "GB"}


def background_location(country):
    return COUNTRY_ALIASES.get(country, country)


@lru_cache(maxsize=None)
def activity_index(database_name):
    index = {}
    duplicates = set()
    for activity in bd.Database(database_name):
        key = (activity.get("name"), activity.get("location"))
        if key in index:
            duplicates.add(key)
        else:
            index[key] = activity
    return index, duplicates


def find_unique_activity(database_name, activity_name, location):
    index, duplicates = activity_index(database_name)
    key = (activity_name, location)
    return None if key in duplicates else index.get(key)


def normalize_positive_shares(series):
    series = series.astype(float).clip(lower=0)
    total = series.sum()
    return series / total if total else series


def shrecc_activity_shares(table, country):
    shares = table[country].groupby(level="activityName").sum()
    return normalize_positive_shares(shares[shares > 0])


def background_production_shares(database_name, country):
    activity = find_unique_activity(
        database_name,
        "electricity, high voltage, production mix",
        background_location(country),
    )
    if activity is None:
        return None

    inputs = []
    for exchange in activity.technosphere():
        input_activity = exchange.input
        if (
            input_activity.get("reference product") == "electricity, high voltage"
            and exchange["amount"] > 0
        ):
            inputs.append((input_activity.get("name"), float(exchange["amount"])))

    if not inputs:
        return None
    shares = pd.DataFrame(inputs, columns=["technology", "share"]).groupby(
        "technology"
    )["share"].sum()
    return normalize_positive_shares(shares)

In [ ]:
activity_index.cache_clear()
mix_rows = []
mix_comparison_gaps = []

for year in YEARS:
    for country in COUNTRIES:
        shrecc_shares = shrecc_activity_shares(annual_tables[year], country)
        background_shares = background_production_shares(
            BACKGROUND_DATABASES[year], country
        )

        if background_shares is None:
            mix_comparison_gaps.append(
                {
                    "year": year,
                    "country": country,
                    "missing": "background production mix",
                }
            )
            continue

        for model, shares in (
            ("SHRECC", shrecc_shares),
            ("Background", background_shares),
        ):
            mix_rows.extend(
                {
                    "year": year,
                    "country": country,
                    "model": model,
                    "background": BACKGROUND_LABELS[year],
                    "technology": technology,
                    "share": share,
                }
                for technology, share in shares.items()
            )

mix_comparison = pd.DataFrame(mix_rows)
mix_comparison.to_csv(OUTPUT_DIR / "technology_mix_comparison.csv", index=False)

mix_comparison_gaps = pd.DataFrame(mix_comparison_gaps)
display(mix_comparison_gaps if not mix_comparison_gaps.empty else "No comparison gaps")
display(mix_comparison.head())

In [ ]:
def plot_mix_comparison(country, top_n=12):
    country_data = mix_comparison.loc[mix_comparison["country"] == country].copy()
    if country_data.empty:
        raise ValueError(f"No mix-comparison data for {country}")

    leading_technologies = (
        country_data.groupby("technology")["share"]
        .sum()
        .nlargest(top_n)
        .index
    )
    country_data["technology_plot"] = country_data["technology"].where(
        country_data["technology"].isin(leading_technologies),
        "REST",
    )
    technologies = list(leading_technologies) + ["REST"]
    colors = dict(
        zip(technologies, plt.cm.tab20(np.linspace(0, 1, len(technologies))))
    )

    fig, axes = plt.subplots(
        len(YEARS),
        1,
        figsize=(12, 10),
        sharex=True,
        squeeze=False,
    )
    axes = axes[:, 0]

    for year, ax in zip(YEARS, axes):
        yearly = country_data.loc[country_data["year"] == year]
        plotted = yearly.pivot_table(
            index="model",
            columns="technology_plot",
            values="share",
            aggfunc="sum",
            fill_value=0,
        ).reindex(index=["SHRECC", "Background"], columns=technologies, fill_value=0)
        plotted.plot.barh(
            stacked=True,
            ax=ax,
            color=[colors[column] for column in plotted.columns],
            legend=False,
            width=0.72,
        )
        ax.set(ylabel=str(year), xlim=(0, 1))
        ax.grid(axis="x", alpha=0.2)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=colors[label], label=label)
        for label in technologies
    ]
    fig.legend(
        handles=handles,
        title="Mapped activity",
        loc="center left",
        bbox_to_anchor=(0.79, 0.5),
        frameon=False,
    )
    axes[-1].set_xlabel("Electricity share")
    fig.suptitle(f"Annual electricity composition in {country}")
    fig.subplots_adjust(left=0.12, right=0.77, top=0.94, bottom=0.08, hspace=0.2)
    return fig, axes


plot_mix_comparison(COUNTRY_TO_PLOT);

## Compare climate-change impacts

For every country and year, this section compares 1 kWh from the annual SHRECC foreground activity with 1 kWh from the corresponding ecoinvent or premise `market for electricity, high voltage`. The impact category is EF v3.1 climate change, GWP100.

The calculations are batched by year so only one foreground/background pair needs to be active in each `MultiLCA`.

In [ ]:
def find_foreground_activity(database_name, country, year):
    return find_unique_activity(
        database_name,
        f"Electricity mix in {country}, {year}",
        country,
    )


def find_background_market(database_name, country):
    return find_unique_activity(
        database_name,
        "market for electricity, high voltage",
        background_location(country),
    )


def calculate_lcia_for_year(year):
    demands = {}
    metadata = {}
    gaps = []

    for country in COUNTRIES:
        foreground = find_foreground_activity(
            FOREGROUND_DATABASES[year], country, year
        )
        background = find_background_market(BACKGROUND_DATABASES[year], country)

        for model, activity in (("SHRECC", foreground), ("Background", background)):
            if activity is None:
                gaps.append(
                    {
                        "year": year,
                        "country": country,
                        "missing": model,
                    }
                )
                continue
            label = f"{year}:{country}:{model}"
            demands[label] = {activity.id: 1}
            metadata[label] = (country, model)

    if not demands:
        return [], gaps

    data_objects = bd.get_multilca_data_objs(
        functional_units=demands,
        method_config=METHOD_CONFIG,
    )
    lca = bc.MultiLCA(
        demands=demands,
        method_config=METHOD_CONFIG,
        data_objs=data_objects,
    )
    lca.lci()
    lca.lcia()

    rows = []
    for label, (country, model) in metadata.items():
        score = float(np.asarray(lca.scores[label]).reshape(-1)[0])
        rows.append(
            {
                "year": year,
                "country": country,
                "model": model,
                "background": BACKGROUND_LABELS[year],
                "score": score,
            }
        )

    del lca, data_objects
    collect()
    return rows, gaps

In [ ]:
activity_index.cache_clear()
lcia_rows = []
lcia_gaps = []

for year in YEARS:
    if FOREGROUND_DATABASES[year] not in bd.databases:
        raise ValueError(
            f"Foreground database has not been written: {FOREGROUND_DATABASES[year]}"
        )
    print(f"Calculating EF v3.1 climate change for {year}")
    yearly_rows, yearly_gaps = calculate_lcia_for_year(year)
    lcia_rows.extend(yearly_rows)
    lcia_gaps.extend(yearly_gaps)

lcia_results = pd.DataFrame(lcia_rows)
lcia_results.to_csv(OUTPUT_DIR / "climate_change_results.csv", index=False)

lcia_gaps = pd.DataFrame(lcia_gaps)
display(lcia_gaps if not lcia_gaps.empty else "No LCIA activity gaps")

lcia_comparison = lcia_results.pivot_table(
    index=["year", "country"],
    columns="model",
    values="score",
).rename_axis(columns=None)
lcia_comparison["difference"] = (
    lcia_comparison["SHRECC"] - lcia_comparison["Background"]
)
lcia_comparison["relative difference (%)"] = (
    100 * lcia_comparison["difference"] / lcia_comparison["Background"]
)
lcia_comparison.to_csv(OUTPUT_DIR / "climate_change_comparison.csv")
display(lcia_comparison)

In [ ]:
method_unit = bd.Method(CLIMATE_METHOD).metadata.get("unit", "kg CO2-eq")

country_lcia = (
    lcia_results.loc[lcia_results["country"] == COUNTRY_TO_PLOT]
    .pivot(index="year", columns="model", values="score")
    .reindex(YEARS)
)
ax = country_lcia.plot.bar(
    figsize=(9, 4.5),
    color={"SHRECC": "#2F6B9A", "Background": "#777777"},
)
ax.set(
    title=f"Climate-change impact of electricity consumed in {COUNTRY_TO_PLOT}",
    xlabel="Model year",
    ylabel=f"{method_unit} per kWh",
)
ax.legend(frameon=False)
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()

relative_difference = (
    lcia_comparison["relative difference (%)"]
    .unstack("year")
    .reindex(index=COUNTRIES, columns=YEARS)
)
finite_values = relative_difference.to_numpy()[
    np.isfinite(relative_difference.to_numpy())
]
color_limit = max(abs(finite_values).max(), 1) if finite_values.size else 1

fig, ax = plt.subplots(figsize=(9, 12))
image = ax.imshow(
    relative_difference,
    aspect="auto",
    cmap="RdBu_r",
    vmin=-color_limit,
    vmax=color_limit,
)
ax.set(
    title="SHRECC climate-change difference relative to the background market",
    xlabel="Model year",
    ylabel="Consumer country",
    xticks=np.arange(len(YEARS)),
    yticks=np.arange(len(COUNTRIES)),
    xticklabels=YEARS,
    yticklabels=COUNTRIES,
)
colorbar = fig.colorbar(image, ax=ax, pad=0.02)
colorbar.set_label("Relative difference (%)")
plt.tight_layout()

## Saved outputs

The restartable annual tables and complete comparison results are written under `notebooks/export/annual_validation`:

- `annual_mix_<year>.csv`: mapped SHRECC annual mix for every consumer country.
- `technology_mix_comparison.csv`: long-form SHRECC/background composition comparison.
- `climate_change_results.csv`: individual EF v3.1 climate-change scores.
- `climate_change_comparison.csv`: absolute and relative SHRECC/background differences.